In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

In [0]:
select_cols_out = [
    "MBRSHP_NBR",
    "MBRSHP_SID",
    "MAIL_ID",
    "EMAIL_SUBJECT",
    "EMAIL_NAME",
    "FIRST_BOUNCE_DATE",
    "LAST_BOUNCE_DATE",
    "BOUNCE_TOTAL",
    "FIRST_SEND_DATE",
    "LAST_SEND_DATE",
    "SEND_TOTAL",
    "FIRST_OPEN_DATE",
    "LAST_OPEN_DATE",
    "OPEN_TOTAL",
    "FIRST_CLICK_DATE",
    "LAST_CLICK_DATE",
    "CLICK_TOTAL",
    "FIRST_UNSUB_DATE",
    "LAST_UNSUB_DATE",
    "UNSUB_TOTAL",
    "ID",
]

### Transform 

In [0]:
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_table_validator(
    silver_master_member_extended, 
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

In [0]:
email = spark.table(bronze_master_email)#.where(f.length(f.col("MBRSHP_NBR")) == 11)

email = email.withColumn("ID", f.monotonically_increasing_id())

member_extended = spark.table(silver_master_member_extended)

df_email = email.join(member_extended, "MBRSHP_NBR", "inner").select(
    *select_cols_out
)

df_email.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'email', config_validation, df_email, stats_etl_path
    )

In [0]:
# evaluate repartition:  original code is like:
# email.repartition("FIRST_SEND_DATE").write.parquet(
#         data_paths["intermediate"]["email"],
#         partitionBy="FIRST_SEND_DATE",
#         mode="overwrite",
#     )

### Merge

In [0]:
df_email.write.mode('overwrite').saveAsTable(silver_master_email)

if archive_flag:
    save_archive(df_email, silver_master_email_archive, run_as_date)